In [2]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import pyarrow.parquet as pq
from tabulate import tabulate

In [3]:
tabler=pq.read_table("clean_auto_data.parquet")
df=tabler.to_pandas()

In [3]:
load_dotenv()
phost=os.getenv("DB_HOST")
pport=os.getenv("DB_PORT")
puser=os.getenv("DB_USER")
ppass=os.getenv("DB_PASS")
pname=os.getenv("DB_NAME")

In [4]:
db_url=f"postgresql+psycopg://{puser}:{ppass}@{phost}:{pport}/{pname}"

In [5]:
engine=create_engine(url=db_url, echo=False, pool_size=5, max_overflow=10)

In [ ]:
df.to_sql("cars", engine, if_exists="replace", index=False)

In [6]:
df=pd.read_sql_query('SELECT * FROM cars', con=engine)

### Finding Undervalued Market Deals

**Business Context:**
Find cars priced significantly lower than the market average for their specific model and production year to spot profitable options for resale or quick purchase.

**Requirements:**
Write an SQL query using a Common Table Expression (CTE) to calculate the average price, average mileage, and total number of listings for each brand, model, and production year (excluding cars where the model is 'unknown'). In the main query, filter for listings that belong to a group of at least 5 cars and are priced more than 20% below their group's average. Include calculated columns showing the potential savings in both currency (BYN) and percentage, and sort the results from the biggest savings to the smallest.

In [12]:
with engine.begin() as conn:
    query="""
        WITH market_benchmarks AS(
            SELECT 
                brand, model, year, price, mileage,
                COUNT(*) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS group_count,
                AVG(cars.price) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS avg_price,
                AVG(cars.mileage) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS avg_mileage
            FROM cars
            WHERE model!='unknown'
        )
        SELECT brand, model, year, price, ROUND(avg_price, 2) as avg_price, ROUND(avg_mileage, 2) AS avg_mileage, group_count, ROUND(avg_price - price, 2) AS byn_profit, ROUND((avg_price - price)*100/avg_price, 2) AS precent_profit  FROM market_benchmarks WHERE group_count>=5 AND price<avg_price*0.8 ORDER BY byn_profit DESC;
    """
    res=conn.execute(text(query))
    print(tabulate(res.fetchall(), headers=res.keys(), tablefmt="psql"))

    # --- Example output in Jupyter Notebook ---
    
    # df_res=pd.read_sql(text(query), con=engine)
    # display(df_res)

    # ------------------------------------------

+------------+------------------+--------+---------+-------------+---------------+---------------+--------------+------------------+
| brand      | model            |   year |   price |   avg_price |   avg_mileage |   group_count |   byn_profit |   precent_profit |
|------------+------------------+--------+---------+-------------+---------------+---------------+--------------+------------------|
| Chevrolet  | Equinox          |   2024 |   54310 |     76481.1 |       22583.9 |            21 |     22171    |            28.99 |
| Peugeot    | 3008             |   2011 |   17258 |     28843.3 |      247459   |             6 |     11585.3  |            40.17 |
| BMW        | 5 серия          |   2002 |   16384 |     26801.3 |      430514   |             6 |     10417.3  |            38.87 |
| BMW        | X5               |   2002 |   20935 |     30973   |      413382   |             6 |     10038    |            32.41 |
| Volkswagen | Touran           |   2012 |   27003 |     37016   |   

### Top 3 Most Expensive Cars per Brand

**Business Context:**
Segment the most expensive listings for each car brand without collapsing the table to highlight premium offers across the market.

**Requirements:**
Write an SQL query using a Common Table Expression (CTE) to rank all listings within each brand from the highest price to the lowest, ensuring cars with the same price share the same rank without skipping subsequent numbers. Then, filter the results in the outer query to display only the top 3 price positions for each brand, showing the brand, model, price, production year, and rank.

In [15]:
with engine.begin() as conn:
    query="""
        WITH ranked_cars AS(
            SELECT
                brand,
                model,
                price,
                year,
                DENSE_RANK() OVER (
                    PARTITION BY brand
                    ORDER BY price DESC
                ) AS ranking
            FROM cars
        )

        SELECT brand, model, price, year, ranking FROM ranked_cars WHERE ranking<=3;
    """

    res=conn.execute(text(query))
    print(tabulate(res.fetchall(), headers=res.keys(), tablefmt="psql"))

    # --- Example output in Jupyter Notebook ---

    # df_res=pd.read_sql(text(query), con=engine)
    # pd.set_option('display.max_rows', None)
    # pd.set_option('display.max_colwidth', None)
    # display(df_res)

    # ------------------------------------------

+------------------+--------------------+---------+--------+-----------+
| brand            | model              |   price |   year |   ranking |
|------------------+--------------------+---------+--------+-----------|
| Acura            | RDX                |   95271 |   2021 |         1 |
| Acura            | Integra            |   84045 |   2024 |         2 |
| Acura            | Integra            |   79493 |   2024 |         3 |
| Alfa Romeo       | Stelvio            |   94664 |   2022 |         1 |
| Alfa Romeo       | Stelvio            |   94664 |   2022 |         1 |
| Alfa Romeo       | Stelvio            |   78583 |   2021 |         2 |
| Alfa Romeo       | Stelvio            |   75701 |   2017 |         3 |
| Arcfox           | T1                 |   59923 |   2025 |         1 |
| Audi             | A7                 |  109531 |   2017 |         1 |
| Audi             | A6                 |  108014 |   2020 |         2 |
| Audi             | Q7                 |  108014 |

### Market Age Distribution and Cumulative Totals

**Business Context:**
Understand how the market volume is distributed by production year and see what percentage of all listed cars are older than a specific year.

**Requirements:**
Write an SQL query using Common Table Expressions (CTEs) to group the cars by production year (ignoring records with missing year or price) and calculate the number of listings, median price, and average price for each year. Then, calculate the running total of cars from the oldest years to the newest, along with the cumulative percentage of the total market volume, and order the final results by production year.

In [8]:
with engine.begin() as conn:
    query="""
        WITH yearly_stats AS(
            SELECT year,  COUNT(*) as cars_count, PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price) AS median,
            AVG(price) AS avg_price
            FROM cars
            WHERE year IS NOT NULL AND price IS NOT NULL
            GROUP BY year
        ), cumulative_sum AS(
            SELECT year, cars_count, median, avg_price, SUM(cars_count) OVER (ORDER BY year ASC) AS all_cars_count, SUM(cars_count) OVER (ORDER BY year ASC) * 100.0/SUM(cars_count) OVER() AS cumulative
            FROM yearly_stats
        )
        SELECT * FROM cumulative_sum ORDER BY year ASC;
    """

    res=conn.execute(text(query))
    print(tabulate(res.fetchall(), headers=res.keys(), tablefmt="psql"))

    # --- Example output in Jupyter Notebook ---

    # df_res=pd.read_sql(text(query), con=engine)
    # pd.set_option('display.max_rows', None)
    # pd.set_option('display.max_colwidth', None)
    # display(df_res)

    # ------------------------------------------

+--------+--------------+----------+-------------+------------------+--------------+
|   year |   cars_count |   median |   avg_price |   all_cars_count |   cumulative |
|--------+--------------+----------+-------------+------------------+--------------|
|   1986 |            2 |   8261   |      8261   |                2 |    0.0447427 |
|   1988 |            2 |  10164.5 |     10164.5 |                4 |    0.0894855 |
|   1992 |            3 |  10847   |     10543.7 |                7 |    0.1566    |
|   1993 |            1 |  20025   |     20025   |                8 |    0.178971  |
|   1994 |            5 |  12136   |     11165.2 |               13 |    0.290828  |
|   1995 |            4 |  14943   |     16915   |               17 |    0.380313  |
|   1996 |           15 |  11378   |     18679.9 |               32 |    0.715884  |
|   1997 |           14 |  14791.5 |     14943.1 |               46 |    1.02908   |
|   1998 |           34 |  11530   |     14251.4 |               

### Mileage Anomalies and High Wear Detection

**Business Context:**
Find cars with very high mileage for their production year to spot high wear risks on the market.

**Requirements:**
Write an SQL query using Common Table Expressions (CTEs) to compare each car's mileage with the average mileage for its production year (only for cars made up to 2025). Calculate how many times a car's mileage is higher than the yearly average. Mark cars where the mileage is more than 80% above the yearly average as 'High Wear / Taxi Risk' (and 'Normal' for others), then show the top 15 cars with the highest ratio.*

In [10]:
with engine.begin() as conn:
    query="""
        WITH et_mileage AS(
            SELECT brand, model, year, mileage, AVG(mileage) OVER(PARTITION BY year) AS avg_mileage FROM cars WHERE year<=2025
        ),
        mileage_d AS(
            SELECT brand, model, year, mileage, ROUND(avg_mileage, 2) AS avg_year_mileage, ROUND(mileage/NULLIF(avg_mileage, 0), 2) AS coefficent,
            CASE
                WHEN mileage>avg_mileage*1.8
                    THEN 'High Wear / Taxi Risk'
                    ELSE 'Normal'
                END AS status
            FROM et_mileage
        )
        SELECT * FROM mileage_d ORDER BY coefficent DESC LIMIT 15;
    """

    res=conn.execute(text(query))
    print(tabulate(res.fetchall(), headers=res.keys(), tablefmt="psql"))

    # --- Example output in Jupyter Notebook ---

    # df_res=pd.read_sql(text(query), con=engine)
    # pd.set_option('display.max_rows', None)
    # pd.set_option('display.max_colwidth', None)
    # display(df_res)

    # ------------------------------------------

+------------+-----------------------+--------+-----------+--------------------+--------------+-----------------------+
| brand      | model                 |   year |   mileage |   avg_year_mileage |   coefficent | status                |
|------------+-----------------------+--------+-----------+--------------------+--------------+-----------------------|
| GMC        | Terrain               |   2025 |     51948 |            8293.22 |         6.26 | High Wear / Taxi Risk |
| Buick      | Encore GX             |   2024 |    102015 |           22759.3  |         4.48 | High Wear / Taxi Risk |
| Nissan     | Rogue                 |   2025 |     32369 |            8293.22 |         3.9  | High Wear / Taxi Risk |
| Nissan     | Rogue                 |   2025 |     29629 |            8293.22 |         3.57 | High Wear / Taxi Risk |
| Chevrolet  | Equinox               |   2023 |    249044 |           71242.6  |         3.5  | High Wear / Taxi Risk |
| Volkswagen | Jetta                 |  